# Amazon Laptop Scraping

In [2]:
# import libraries
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd


In [3]:
#!pip install beautifulsoup4 requests pandas

In [4]:
# take the url of the page
url = "https://www.amazon.in/s?k=laptops&crid=3R9WE5K6F9ZLA&sprefix=laptops%2Caps%2C522&ref=nb_sb_noss_2"

In [5]:
# create the request header

headers = {
    "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36"
}


In [6]:
# send the request
response = requests.get(url, headers=headers)

# check if the request was successful
response.status_code

200

In [7]:
# show the content of the page
response.content

b'<!doctype html><html lang="en-in" class="a-no-js" data-19ax5a9jf="dingo"><!-- sp:feature:head-start -->\n<head><script>var aPageStart = (new Date()).getTime();</script><meta charset="utf-8"/>\n<!-- sp:end-feature:head-start -->\n<!-- sp:feature:csm:head-open-part1 -->\n\n<script type=\'text/javascript\'>var ue_t0=ue_t0||+new Date();</script>\n<!-- sp:end-feature:csm:head-open-part1 -->\n<!-- sp:feature:cs-optimization -->\n<meta http-equiv=\'x-dns-prefetch-control\' content=\'on\'>\n<link rel="preconnect" href="https://images-eu.ssl-images-amazon.com" crossorigin>\n<link rel="preconnect" href="https://m.media-amazon.com" crossorigin>\n<!-- sp:end-feature:cs-optimization -->\n<!-- sp:feature:csm:head-open-part2 -->\n<script type=\'text/javascript\'>\nwindow.ue_ihb = (window.ue_ihb || window.ueinit || 0) + 1;\nif (window.ue_ihb === 1) {\n\nvar ue_csm = window,\n    ue_hob = +new Date();\n(function(d){var e=d.ue=d.ue||{},f=Date.now||function(){return+new Date};e.d=function(b){return f()

In [8]:
# beautiful soup object

bs = BeautifulSoup(response.content, "html.parser")
print(bs)

<!DOCTYPE html>
<html class="a-no-js" data-19ax5a9jf="dingo" lang="en-in"><!-- sp:feature:head-start -->
<head><script>var aPageStart = (new Date()).getTime();</script><meta charset="utf-8"/>
<!-- sp:end-feature:head-start -->
<!-- sp:feature:csm:head-open-part1 -->
<script type="text/javascript">var ue_t0=ue_t0||+new Date();</script>
<!-- sp:end-feature:csm:head-open-part1 -->
<!-- sp:feature:cs-optimization -->
<meta content="on" http-equiv="x-dns-prefetch-control"/>
<link crossorigin="" href="https://images-eu.ssl-images-amazon.com" rel="preconnect"/>
<link crossorigin="" href="https://m.media-amazon.com" rel="preconnect"/>
<!-- sp:end-feature:cs-optimization -->
<!-- sp:feature:csm:head-open-part2 -->
<script type="text/javascript">
window.ue_ihb = (window.ue_ihb || window.ueinit || 0) + 1;
if (window.ue_ihb === 1) {

var ue_csm = window,
    ue_hob = +new Date();
(function(d){var e=d.ue=d.ue||{},f=Date.now||function(){return+new Date};e.d=function(b){return f()-(b?0:d.ue_t0)};e.st

In [9]:
# create the empty list to store the data
Data = []

In [10]:
list_data = []

for page in range(1, 30):
    params = {"k": "laptops", "page": page}
    response = requests.get(url, headers=headers, params=params, timeout=30)
    response.raise_for_status()
    bs = BeautifulSoup(response.text, "html.parser")

    product_containers = bs.select('div[data-component-type="s-search-result"]')

    for product in product_containers:

        title_tag = product.select_one("h2 span")
        if not title_tag:
            continue

        #Step 1:Extract the Title
        title = title_tag.get_text(" ", strip=True)

        # Step 2: Extract the Price
        price_tag = product.select_one("span.a-price-whole")

        # Step 3: Extract the Rating
        rating_tag = product.select_one("span.a-size-small.a-color-base")
        match = re.match(r"([A-Za-z]+)", title)


        #Step 5: Extract the Brand Name
        brand = match.group(1) if match else "Unknown"
        
        #Step 6: Extract the RAM
        match = re.search(r"(\d+GB|RAM\s*(\d+GB)?|DDR\d?\s*(\d+GB)?|LPDDR\d?\s*(\d+GB)?)", title, re.IGNORECASE)
        ram = match.group(1) if match else "N/A"

        #Step 7: Extract the Storage 
        match = re.search(r"(\d+)\s*(GB|TB)\s*(?:SSD|Storage|HDD)", title, re.IGNORECASE)
        ssd_storage = f"{match.group(1)}{match.group(2)}" if match else "N/A"

        #Step 8: Extract the Color
        match = re.search(r"\b(Black|White|Silver|Gray|Grey|Red|Blue|Green|Yellow|Pink|Purple|Gold|Bronze|Rose Gold|Indigo|Glacier)\b", title, re.IGNORECASE)
        color = match.group(1) if match else "N/A"
       
        # Step 9: Extract the processor information
        # Step 7: Extract the processor (Intel, AMD, Apple M/A chip, Snapdragon, MediaTek, etc)
        processor = "N/A"
        # Try to match Apple M series (M1, M2, M3, M4, M5, etc.)
        match = re.search(r"Apple\s+M(\d+)", title, re.IGNORECASE)
        if match:
            processor = f"Apple M{match.group(1)}"
        else:
            # Try to match Apple A series (A18, A17, A16, etc.)
            match = re.search(r"Apple\s+A(\d+)", title, re.IGNORECASE)
            if match:
                processor = f"Apple A{match.group(1)}"
            else:
                # Try other processor keywords
                processor_keywords = ["Intel", "AMD", "Snapdragon", "MediaTek", "Celeron"]
                for keyword in processor_keywords:
                    if re.search(rf"\b{keyword}\b", title, re.IGNORECASE):
                        processor = keyword
                        break

        # Append the extracted data to the list
        list_data.append({
            "title": title,
            "price": price_tag.get_text(strip=True) if price_tag else "N/A",
            "rating": rating_tag.get_text(strip=True) if rating_tag else "N/A",
            "brand": brand,
            "ram": ram,
            "storage": ssd_storage,
            "color": color,
            "processor": processor
            })

In [11]:
# display the data item 

for item in list_data:
    print(f"Title: {item['title']}")
    print(f"Price: {item['price']}")
    print(f"Rating: {item['rating']}")
    print(f"Brand: {item['brand']}")
    print(f"RAM: {item['ram']}")
    print(f"Storage: {item['storage']}")
    print(f"Color: {item['color']}")
    print(f"Processor: {item['processor']}")

    print("-" * 50)

Title: HP 15 (i5 14th Gen), Intel Core 5, 16GB RAM (Upgradeable), 512GB SSD, FHD, Anti-Glare, 15.6''/39.6cm, Win11, M365 Basic(1yr), Office24, Silver,1.59kg, fd0682tu, FHD Camera w/Shutter, Backlit Laptop
Price: 75,990
Rating: 3.6
Brand: HP
RAM: 16GB
Storage: 512GB
Color: Silver
Processor: Intel
--------------------------------------------------
Title: HP Omnibook 3, Snapdragon X Processor 45 Tops (16GB LPDDR5x,512GB SSD) 2K WUXGA, 14''/35.6cm, Win 11, M365*Office 24,Silver,1.42kg, hz0026QU/ hz0024QU, Lighter mini Charger, FHD IR Camera, AI Laptop
Price: 69,990
Rating: 3.2
Brand: HP
RAM: 16GB
Storage: 512GB
Color: Silver
Processor: Snapdragon
--------------------------------------------------
Title: Lenovo V15 G4 AMD Athlon Silver 7120U Laptop 8GB LPDDR5 Ram, 512 GB SSD PCIe, Windows 11 Lifetime Validity,15.6" FHD Screen, AMD Radeon 610M, Silver, 1 Year Brand Warranty
Price: 44,999
Rating: 4.0
Brand: Lenovo
RAM: 8GB
Storage: 512GB
Color: Silver
Processor: AMD
--------------------------

In [12]:
# check the length of the data
print(f"Total number of laptops scraped: {len(list_data)}")

Total number of laptops scraped: 557


In [15]:
# display the match
print(brand)

Apple


In [16]:
# save the data to a csv file
df = pd.DataFrame(list_data)

In [17]:
df.to_csv("Amazon_laptops.csv",index = False)

In [ ]:
df